# Ride-hailing dataset: preprocessing for ML/DL

Goal: turn `df_normalized.csv` into a fully numeric dataset with no missing values.

Steps
1. Load and inspect
2. Remove rows that are not real rides
3. Remove duplicate rows
4. Clean and consolidate categorical values
5. Fix the distance column
6. Date and time features
7. Location features
8. Choose the target and remove leaking columns
9. Drop redundant columns and one-hot encode
10. Scale numeric columns
11. Validate and save

## 0. Imports and settings

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

OUTPUT = "ml_ready.csv"

# Choose what you want to predict:
#   "fare_amount" -> regression
#   "status"      -> classification (ride outcome)
TARGET = "fare_amount"

## 1. Load and inspect

In [3]:
df = pd.read_csv(r"C:\Users\lenovo\OneDrive\Desktop\Folder\cab_bike_auto_fare_prediction\df_normalized.csv")
print(df.shape)
df.head()

(385324, 21)


,source_dataset,date,hour,vehicle_type,pickup_location,drop_location,distance_km,fare_amount,payment_method,status,...,customer_rating,year,month,day,weekday,state,season,weather,phase_of_day,is_phase_imputed
0,bengaluru_ola,2024-01-28,6.0,auto,Area-3,Area-2,28.50,868.06,wallet,success,...,4.4,2024.0,1.0,28.0,Sunday,Karnataka,winter,not_applicable,morning,False
1,bengaluru_ola,2024-01-17,21.0,cab_ac,Area-38,Area-26,25.18,348.04,card,success,...,4.7,2024.0,1.0,17.0,Wednesday,Karnataka,winter,not_applicable,night,False
2,bengaluru_ola,2024-01-30,0.0,bike,Area-47,Area-8,17.67,56.33,upi,success,...,3.3,2024.0,1.0,30.0,Tuesday,Karnataka,winter,not_applicable,midnight,False
3,bengaluru_ola,2024-01-22,8.0,cab_suv,Area-11,Area-35,24.94,1971.38,upi,success,...,3.7,2024.0,1.0,22.0,Monday,Karnataka,winter,not_applicable,morning,False
4,bengaluru_ola,2024-01-12,7.0,cab_ac,Area-43,Area-42,13.15,1130.15,card,success,...,3.0,2024.0,1.0,12.0,Friday,Karnataka,winter,not_applicable,morning,False


In [4]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nRows per source:")
print(df["source_dataset"].value_counts())

Missing values per column:
date               1270
hour             139487
drop_location       928
year               1270
month              1270
day                1270
weekday            1270
dtype: int64

Exact duplicate rows: 7329

Rows per source:
source_dataset
ncr_events                 109329
bookings                   103024
indore_ola                 100000
rides_data                  38217
bengaluru_ola               33484
delhi_notification_2023       928
aru_dto_2019                  342
Name: count, dtype: int64


## 2. Remove rows that are not real rides

`delhi_notification_2023` and `aru_dto_2019` are official rate cards / fixed tariffs
(status is `official_rate_card`, `fixed`, `per_seat`, `per_day`; no date, no drop location,
constant ratings). They are not trips, so they would only add noise. Removing them also
removes every NaN in `date`, `weekday` and `drop_location`.

In [5]:
NON_TRIP_SOURCES = ["delhi_notification_2023", "aru_dto_2019"]

mask = df["source_dataset"].isin(NON_TRIP_SOURCES)
print("Dropping", mask.sum(), "rate-card / tariff rows")
df = df.loc[~mask].copy()
df.shape

Dropping 1270 rate-card / tariff rows


(384054, 21)

## 3. Remove duplicate rows

All duplicates are in `ncr_events`. There is no ID column, but with date, hour, locations,
distance and fare all identical, these are almost certainly repeated records and not
coincidences.

In [6]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Dropped", n_before - len(df), "duplicate rows ->", df.shape)

Dropped 7329 duplicate rows -> (376725, 21)


## 4. Clean and consolidate categorical values

* `payment_method` has 11 labels that overlap (card / credit card / debit card, ...).
  Group them into cash, card, upi, wallet.
* `status`: `success` and `completed` mean the same thing (different source files).

In [7]:
for c in ["source_dataset", "vehicle_type", "payment_method", "status", "state", "season", "weather"]:
    df[c] = df[c].astype(str).str.strip().str.lower()

PAYMENT_MAP = {
    "cash": "cash",
    "card": "card", "credit card": "card", "debit card": "card",
    "upi": "upi", "gpay": "upi", "qr scan": "upi",
    "wallet": "wallet", "uber wallet": "wallet", "paytm": "wallet", "amazon pay": "wallet",
}
STATUS_MAP = {
    "success": "completed", "completed": "completed",
    "incomplete": "incomplete",
    "canceled by customer": "canceled by customer",
    "canceled by driver": "canceled by driver",
    "driver not found": "driver not found",
}

df["payment_group"] = df["payment_method"].map(PAYMENT_MAP)
df["status"] = df["status"].map(STATUS_MAP)

assert df["payment_group"].notna().all(), "unmapped payment_method"
assert df["status"].notna().all(), "unmapped status"
print(df["payment_group"].value_counts(), "\n")
print(df["status"].value_counts())

payment_group
cash      207998
upi        99261
wallet     39674
card       29792
Name: count, dtype: int64 

status
completed               290577
canceled by driver       36478
incomplete               22004
canceled by customer     17542
driver not found         10124
Name: count, dtype: int64


## 5. Fix the distance column

`distance_km == 0` only happens for cancelled bookings, where the ride never started. The
value is really "not recorded", so treat it as missing, keep a flag, and fill with the
median distance of the same source.

In [8]:
df["distance_missing"] = (df["distance_km"] <= 0).astype(int)
df.loc[df["distance_km"] <= 0, "distance_km"] = np.nan
df["distance_km"] = df["distance_km"].fillna(
    df.groupby("source_dataset")["distance_km"].transform("median")
)
print("Missing-distance rows flagged:", df["distance_missing"].sum())
print("NaN left in distance_km:", df["distance_km"].isna().sum())

Missing-distance rows flagged: 39057
NaN left in distance_km: 0


## 6. Date and time features

* `date` comes in two formats (`YYYY-MM-DD` and `YYYY-MM-DD HH:MM:SS`), so parse the first 10 characters.
* `hour` is missing for about 36% of rows. `phase_of_day` is a fixed function of hour
  (midnight 0-3, before_sunrise 4-5, morning 6-11, afternoon 12-15, evening 16-18, night 19-23),
  so fill the hour with the typical hour of that phase and keep `is_phase_imputed`.
* Cyclic values (hour, month, day, weekday) are encoded as sin/cos so that 23:00 is close to 00:00.

In [9]:
date = pd.to_datetime(df["date"].astype(str).str[:10], format="%Y-%m-%d")
df["year"] = date.dt.year
df["month"] = date.dt.month
df["day"] = date.dt.day
df["dow"] = date.dt.dayofweek          # 0 = Monday
df["is_weekend"] = (df["dow"] >= 5).astype(int)

# impute hour from phase_of_day
df["is_phase_imputed"] = df["is_phase_imputed"].astype(int)
phase_hour = df.loc[df["hour"].notna()].groupby("phase_of_day")["hour"].median().round()
print(phase_hour)
df["hour"] = df["hour"].fillna(df["phase_of_day"].map(phase_hour))
assert df["hour"].notna().all()

def add_cyclical(data, col, period):
    data[f"{col}_sin"] = np.sin(2 * np.pi * data[col] / period)
    data[f"{col}_cos"] = np.cos(2 * np.pi * data[col] / period)

add_cyclical(df, "hour", 24)
add_cyclical(df, "month", 12)
add_cyclical(df, "day", 31)
add_cyclical(df, "dow", 7)

phase_of_day
afternoon         14.0
before_sunrise     5.0
evening           17.0
midnight           1.0
morning            9.0
night             21.0
Name: hour, dtype: float64


## 7. Location features

There are about 12.8k unique pickup locations. One-hot encoding would create about 13k
columns, so use the log of how often each location appears, plus a flag for pickup == drop.

In [10]:
df["pickup_location"] = df["pickup_location"].astype(str).str.strip().str.lower()
df["drop_location"]   = df["drop_location"].astype(str).str.strip().str.lower()

counts = pd.concat([df["pickup_location"], df["drop_location"]]).value_counts()
df["pickup_freq"] = np.log1p(df["pickup_location"].map(counts))
df["drop_freq"]   = np.log1p(df["drop_location"].map(counts))
df["same_pickup_drop"] = (df["pickup_location"] == df["drop_location"]).astype(int)

## 8. Choose the target and remove leaking columns

For `status` prediction, three things leak the answer, so they are removed:
* ratings are filled with a constant 4.2 for every non-completed ride
* `distance_km` is 0 for every cancelled booking
* `weather == "unknown"` appears only on cancelled bookings

In [11]:
STATUS_CODES = {
    "completed": 0, "incomplete": 1, "canceled by customer": 2,
    "canceled by driver": 3, "driver not found": 4,
}
LEAKY_FOR_STATUS = ["driver_rating", "customer_rating", "distance_km", "distance_missing", "weather"]

if TARGET == "fare_amount":
    y = df["fare_amount"].astype(float)
    drop_cols = ["fare_amount"]
else:
    y = df["status"].map(STATUS_CODES).astype(int)
    drop_cols = list(LEAKY_FOR_STATUS)

y.describe() if TARGET == "fare_amount" else y.value_counts()

count    376725.000000
mean        569.160264
std         456.177620
min          50.000000
25%         261.000000
50%         436.000000
75%         748.000000
max        4277.000000
Name: fare_amount, dtype: float64

## 9. Drop redundant columns and one-hot encode

* raw date/time columns are replaced by the sin/cos features
* `phase_of_day` is fully determined by hour
* `weekday` duplicates `dow`
* `state` is fully determined by `source_dataset`
* location names are replaced by the frequency features

In [12]:
drop_cols += [
    "date", "hour", "month", "day", "dow",
    "phase_of_day", "weekday",
    "payment_method", "status",
    "state",
    "pickup_location", "drop_location",
]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])

ONE_HOT_COLS = ["source_dataset", "vehicle_type", "payment_group", "season", "weather"]
ohe_cols = [c for c in ONE_HOT_COLS if c in X.columns]
X = pd.get_dummies(X, columns=ohe_cols, drop_first=True, dtype=int)
print(X.shape)
X.head()

(376725, 36)


,distance_km,driver_rating,customer_rating,year,is_phase_imputed,distance_missing,is_weekend,hour_sin,hour_cos,month_sin,...,vehicle_type_premier_sedan,payment_group_cash,payment_group_upi,payment_group_wallet,season_post_monsoon,season_summer,season_winter,weather_raining,weather_sunny,weather_unknown
0,28.50,4.4,4.4,2024,0,0,1,1.000000,6.123234e-17,0.5,...,0,0,0,1,0,0,1,0,0,0
1,25.18,4.5,4.7,2024,0,0,0,-0.707107,7.071068e-01,0.5,...,0,0,0,0,0,0,1,0,0,0
2,17.67,3.6,3.3,2024,0,0,0,0.000000,1.000000e+00,0.5,...,0,0,1,0,0,0,1,0,0,0
3,24.94,3.2,3.7,2024,0,0,0,0.866025,-5.000000e-01,0.5,...,0,0,1,0,0,0,1,0,0,0
4,13.15,3.1,3.0,2024,0,0,0,0.965926,-2.588190e-01,0.5,...,0,0,0,0,0,0,1,0,0,0


## 10. Scale numeric columns

Standardise the continuous columns (mean 0, std 1). Binary, one-hot and sin/cos columns are left as they are.
The target is NOT scaled.

In [13]:
to_scale = [c for c in ["distance_km", "fare_amount", "driver_rating", "customer_rating",
                        "year", "pickup_freq", "drop_freq"] if c in X.columns]
print("Scaling:", to_scale)

scaler = StandardScaler()
X[to_scale] = scaler.fit_transform(X[to_scale])
X[to_scale].describe().round(2)

Scaling: ['distance_km', 'driver_rating', 'customer_rating', 'year', 'pickup_freq', 'drop_freq']


,distance_km,driver_rating,customer_rating,year,pickup_freq,drop_freq
count,376725.00,376725.00,376725.00,376725.00,376725.00,376725.00
mean,-0.00,-0.00,-0.00,-0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.51,-2.51,-2.51,-0.60,-3.53,-3.53
25%,-0.80,-0.50,-0.36,-0.60,-0.10,-0.10
50%,-0.14,0.17,0.07,-0.60,0.55,0.55
75%,0.66,0.40,0.51,1.66,0.57,0.57
max,2.28,1.96,1.80,1.66,1.01,1.01


## 11. Validate and save

In [14]:
out = X.copy()
out["target"] = y.values

assert not out.isna().any().any(), "NaNs remain"
assert np.isfinite(out.to_numpy(dtype=float)).all(), "inf remains"
assert all(pd.api.types.is_numeric_dtype(t) for t in out.dtypes), "non-numeric column remains"

out.to_csv(OUTPUT, index=False, float_format="%.6g")
print(out.shape, "->", OUTPUT)
out.head()

(376725, 37) -> ml_ready.csv


,distance_km,driver_rating,customer_rating,year,is_phase_imputed,distance_missing,is_weekend,hour_sin,hour_cos,month_sin,...,payment_group_cash,payment_group_upi,payment_group_wallet,season_post_monsoon,season_summer,season_winter,weather_raining,weather_sunny,weather_unknown,target
0,0.619647,0.620152,0.505212,-0.60114,0,0,1,1.000000,6.123234e-17,0.5,...,0,0,1,0,0,1,0,0,0,868.06
1,0.362899,0.843521,1.151702,-0.60114,0,0,0,-0.707107,7.071068e-01,0.5,...,0,0,0,0,0,1,0,0,0,348.04
2,-0.217879,-1.166802,-1.865250,-0.60114,0,0,0,0.000000,1.000000e+00,0.5,...,0,1,0,0,0,1,0,0,0,56.33
3,0.344338,-2.060279,-1.003264,-0.60114,0,0,0,0.866025,-5.000000e-01,0.5,...,0,1,0,0,0,1,0,0,0,1971.38
4,-0.567429,-2.283648,-2.511740,-0.60114,0,0,0,0.965926,-2.588190e-01,0.5,...,0,0,0,0,0,1,0,0,0,1130.15


In [17]:
from sklearn.model_selection import train_test_split

In [19]:
X

,distance_km,driver_rating,customer_rating,year,is_phase_imputed,distance_missing,is_weekend,hour_sin,hour_cos,month_sin,...,vehicle_type_premier_sedan,payment_group_cash,payment_group_upi,payment_group_wallet,season_post_monsoon,season_summer,season_winter,weather_raining,weather_sunny,weather_unknown
0,0.619647,0.620152,0.505212,-0.601140,0,0,1,1.000000,6.123234e-17,0.5,...,0,0,0,1,0,0,1,0,0,0
1,0.362899,0.843521,1.151702,-0.601140,0,0,0,-0.707107,7.071068e-01,0.5,...,0,0,0,0,0,0,1,0,0,0
2,-0.217879,-1.166802,-1.865250,-0.601140,0,0,0,0.000000,1.000000e+00,0.5,...,0,0,1,0,0,0,1,0,0,0
3,0.344338,-2.060279,-1.003264,-0.601140,0,0,0,0.866025,-5.000000e-01,0.5,...,0,0,1,0,0,0,1,0,0,0
4,-0.567429,-2.283648,-2.511740,-0.601140,0,0,0,0.965926,-2.588190e-01,0.5,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376720,-0.213239,0.173414,0.074219,1.663505,1,0,0,-0.965926,-2.588190e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376721,-0.166066,-0.496694,-1.003264,1.663505,1,0,0,-0.965926,-2.588190e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376722,-0.866711,0.173414,0.074219,1.663505,1,0,0,-0.707107,7.071068e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376723,-1.327622,0.173414,0.074219,1.663505,1,0,1,-0.707107,7.071068e-01,0.5,...,0,1,0,0,0,0,1,0,0,0


In [20]:
y

0          868.06
1          348.04
2           56.33
3         1971.38
4         1130.15
           ...   
376720     908.00
376721     676.00
376722     780.00
376723    1852.00
376724     268.00
Name: fare_amount, Length: 376725, dtype: float64

In [21]:
X_train,X_test,y_train,y_test =  train_test_split(X,y,test_size= 0.2, random_state=42)

In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
mean_absolute_error,
mean_squared_error,
root_mean_squared_error,
r2_score
)

In [29]:
mlr = LinearRegression()

In [30]:
mlr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](36,)","[ 65.87, -0.57, -0.21,..., 377.67,-170.42, -10.06]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](36,)","['distance_km','driver_rating','customer_rating',...,'weather_raining', 'weather_sunny','weather_unknown']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,976.8
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,36
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,32


In [31]:
##Evaluation
y_pred = mlr.predict(X_test)

In [32]:
mean_absolute_error(y_test,y_pred)

294.20941639003354

In [33]:
mean_squared_error(y_test,y_pred)

171772.3718833329

In [34]:
root_mean_squared_error(y_test,y_pred)

414.4543061464471

In [35]:
r2_score(y_test,y_pred)

0.18471139122815616

In [36]:
X

,distance_km,driver_rating,customer_rating,year,is_phase_imputed,distance_missing,is_weekend,hour_sin,hour_cos,month_sin,...,vehicle_type_premier_sedan,payment_group_cash,payment_group_upi,payment_group_wallet,season_post_monsoon,season_summer,season_winter,weather_raining,weather_sunny,weather_unknown
0,0.619647,0.620152,0.505212,-0.601140,0,0,1,1.000000,6.123234e-17,0.5,...,0,0,0,1,0,0,1,0,0,0
1,0.362899,0.843521,1.151702,-0.601140,0,0,0,-0.707107,7.071068e-01,0.5,...,0,0,0,0,0,0,1,0,0,0
2,-0.217879,-1.166802,-1.865250,-0.601140,0,0,0,0.000000,1.000000e+00,0.5,...,0,0,1,0,0,0,1,0,0,0
3,0.344338,-2.060279,-1.003264,-0.601140,0,0,0,0.866025,-5.000000e-01,0.5,...,0,0,1,0,0,0,1,0,0,0
4,-0.567429,-2.283648,-2.511740,-0.601140,0,0,0,0.965926,-2.588190e-01,0.5,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
376720,-0.213239,0.173414,0.074219,1.663505,1,0,0,-0.965926,-2.588190e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376721,-0.166066,-0.496694,-1.003264,1.663505,1,0,0,-0.965926,-2.588190e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376722,-0.866711,0.173414,0.074219,1.663505,1,0,0,-0.707107,7.071068e-01,0.5,...,0,1,0,0,0,0,1,0,0,0
376723,-1.327622,0.173414,0.074219,1.663505,1,0,1,-0.707107,7.071068e-01,0.5,...,0,1,0,0,0,0,1,0,0,0


In [37]:
from sklearn.preprocessing import PolynomialFeatures

In [38]:
pl=PolynomialFeatures()

In [49]:
poly_X=pl.fit_transform(X)
poly_y=pl.fit_transform(pd.DataFrame(y))

In [40]:
poly_X

array([[ 1.        ,  0.61964723,  0.62015205, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.        ,  0.36289851,  0.84352127, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.        , -0.21787947, -1.1668017 , ...,  0.        ,
         0.        ,  0.        ],
       ...,
       [ 1.        , -0.86671132,  0.17341361, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.        , -1.32762167,  0.17341361, ...,  0.        ,
         0.        ,  0.        ],
       [ 1.        , -0.68961657,  0.17341361, ...,  0.        ,
         0.        ,  0.        ]], shape=(376725, 703))

In [50]:
mlr.fit(poly_X,y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](703,)","[ -0. , 121.13, -7.69,...,-122.52, 0. , -22.02]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,928.7
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,703
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,444
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](703,)","[3264.94,2105.43,1266.79,..., 0. , 0. , 0. ]"


In [51]:
y_predict = mlr.predict(poly_X)

ValueError: y_true and y_pred have different number of output (3!=1)